In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_colwidth', None)

sys.path.insert(0, str(next((p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / 'utils').exists()), Path().resolve())))
import utils.imports as imports
import utils.features as features
import steps.isolation_forest as isolation_forest

In [2]:
config_path = "config/config.yml"

In [3]:
isolation_forest_preprocess = isolation_forest.IsolationForestPreprocess(config_path=config_path)
raw_data_df, isolation_forest_data_df, isolation_features_list = isolation_forest_preprocess._execute()

data/bank_transactions_data.csv
C:\repos\research\anomaly_bank


In [4]:
isolation_forest_obj = isolation_forest.IsolationForestModel(config_path=config_path,
                                                             raw_df=raw_data_df,
                                                             enriched_df=isolation_forest_data_df,
                                                             feat_columns=isolation_features_list)
anomaly_aggs_df, isolation_df, labeled_raw_df = isolation_forest_obj._execute()

 99%|===================| 2494/2512 [00:18<00:00]        

Iterating through 34 feature counts to find best model...
Model features with best params: anomaly_score_median                                                                                                                                                                                                                                                                                                                                                                                                                                                                         0.016845
anomaly_score_mean                                                                                                                                                                                                                                                                                                                                                                                                                     

In [5]:
# Maximizing median and mean while minimizing stdev

px.line(anomaly_aggs_df, x='feat_columns_count', y=['anomaly_score_median', 'anomaly_score_stdev', 'anomaly_score_mean', 'anomaly_final_score']).show()

In [6]:
# isolation_forest_data_df[['Anomaly', 'AnomalyScore']].head()
px.line(isolation_df.sort_values(by='Anomaly', ascending=True).reset_index(drop=True).reset_index(), 
        x='index', 
        y='AnomalyScore',
        color='Anomaly')

In [7]:
# len(isolation_df.columns), 
len(isolation_forest_data_df.columns), len(raw_data_df.columns), len(labeled_raw_df.columns)

(48, 18, 20)

In [8]:
labeled_raw_df.IsolationForest_Anomaly.value_counts()

IsolationForest_Anomaly
0    2486
1      26
Name: count, dtype: int64

In [9]:
labeled_raw_df.head(2)

,TransactionID,AccountID,TransactionAmount,PreviousTransactionDate,TransactionType,Location,DeviceID,IP_Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,TransactionDate,IP_prefix_1,diff_days,IsolationForest_Anomaly,IsolationForest_AnomalyScore
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08,162,572,0,-0.110228
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35,13,495,0,-0.111175


In [10]:
from sklearn.model_selection import KFold
import numpy as np

In [11]:
n_splits = 5
random_state = 42
column_name = 'MerchantID'
target_name = 'IsolationForest_Anomaly'
alpha = 1.0

df = labeled_raw_df.copy()

In [12]:
kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
new_col = f"{column_name}_te"
df[new_col] = np.nan
global_mean = df[target_name].mean()

for train_idx, val_idx in kf.split(df):
    train, val = df.iloc[train_idx], df.iloc[val_idx]

    # Compute target mean and count per category in training fold
    stats = train.groupby(column_name)[target_name].agg(['mean', 'count']).reset_index()
    
    # Apply smoothing
    stats['smoothed'] = (stats['mean'] * stats['count'] + global_mean * alpha) / (stats['count'] + alpha)
    
    # Map to validation fold
    mapping = dict(zip(stats[column_name], stats['smoothed']))
    df.loc[val_idx, new_col] = df.loc[val_idx, column_name].map(mapping).fillna(global_mean)


In [ ]:
df.head()

,TransactionID,AccountID,TransactionAmount,PreviousTransactionDate,TransactionType,Location,DeviceID,IP_Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,TransactionDate,IP_prefix_1,diff_days,IsolationForest_Anomaly,IsolationForest_AnomalyScore,MerchantID_te
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08,162,572,0,-0.110228,0.040414
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35,13,495,0,-0.111175,0.043928
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04,215,482,0,-0.086843,0.000398
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06,200,548,0,-0.099164,0.000414
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39,65,384,0,-0.066577,0.000383


: 